# 06 — Triangle & Channel Detection Gallery

**Author:** Zeineb Turki  
**Date:** April 2026  
**Phase:** 3.5  

This notebook shows **every** detected triangle and channel pattern individually.
Each chart shows candlesticks, swing-high/low pivot markers, and the fitted
trendlines running from the first pivot to just past the breakout bar.

**Method:** pivot + `linregress` (20-bar window, |r| ≥ 0.9, adjusted intercepts).

In [ ]:
import sys, os
from pathlib import Path

# Force-reload src modules
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')
    _spy_path = os.path.join(PROJ_ROOT, 'data', 'raw', 'spy.csv')
    if not os.path.isfile(_spy_path):
        os.makedirs(os.path.dirname(_spy_path), exist_ok=True)
        import yfinance as yf, pandas as pd
        _spy = yf.download('SPY', start='2010-01-01', end='2026-01-01', auto_adjust=False)
        if isinstance(_spy.columns, pd.MultiIndex): _spy.columns = _spy.columns.droplevel(1)
        _spy = _spy[['Open','High','Low','Close','Volume']]
        if _spy.index.tz is not None: _spy.index = _spy.index.tz_localize(None)
        _spy.index.name = 'Date'
        _spy.to_csv(_spy_path)
else:
    def _find_project_root():
        current = Path.cwd()
        for _ in range(10):
            if (current / 'src').is_dir(): return current
            current = current.parent
        return Path.cwd().parent if (Path.cwd().parent / 'src').is_dir() else Path.cwd()
    PROJ_ROOT = str(_find_project_root())

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.dates as mdates
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

from src.data.load_data import load_spy
from src.patterns.export_patterns import collect_pattern_details
from src.utils.plotting import plot_candlestick, add_trendline, add_event_marker

df = load_spy()
all_details = collect_pattern_details(df)
tri_details = [d for d in all_details if 'triangle' in d['pattern_type']]
ch_details  = [d for d in all_details if 'channel' in d['pattern_type']]
print(f'Triangles: {len(tri_details)}  |  Channels: {len(ch_details)}')

In [ ]:
FORWARD = 10

def plot_detection(det):
    start = det['start_idx']
    end = min(det['end_idx'] + FORWARD, len(df) - 1)
    chart = df.iloc[start:end+1]

    title = f"{det['pattern_type']}  \u2014  {det['event_date'].strftime('%Y-%m-%d')}"
    parts = []
    if 'containment_ratio' in det: parts.append(f"cr={det['containment_ratio']:.0%}")
    if 'r_upper' in det: parts.append(f"|r|=({det['r_upper']:.2f},{det['r_lower']:.2f})")
    if 'pivot_highs' in det: parts.append(f"pivots=({det['pivot_highs']},{det['pivot_lows']})")
    if parts: title += '   ' + '  '.join(parts)

    fig, ax = plt.subplots(figsize=(14, 4.5))
    plot_candlestick(chart, ax=ax, title=title)

    sl_u, ic_u = det['upper_slope'], det['upper_intercept']
    sl_l, ic_l = det['lower_slope'], det['lower_intercept']

    if 'swing_high_idx' in det:
        sh, sl = det['swing_high_idx'], det['swing_low_idx']
        x_end = det['end_idx'] - start + 3
        d1 = mdates.date2num(df.index[min(start + x_end, len(df)-1)])

        # Upper trendline: first swing high -> breakout
        x0 = sh[0] - start
        ax.plot([mdates.date2num(df.index[start+x0]), d1],
                [sl_u*x0+ic_u, sl_u*x_end+ic_u],
                color='red', lw=1.5, alpha=0.8, label='Upper trend')

        # Lower trendline: first swing low -> breakout
        x0 = sl[0] - start
        ax.plot([mdates.date2num(df.index[start+x0]), d1],
                [sl_l*x0+ic_l, sl_l*x_end+ic_l],
                color='blue', lw=1.5, alpha=0.8, label='Lower trend')

        # Swing high markers
        for i,p in zip(sh, det['swing_high_prices']):
            ax.scatter(mdates.date2num(df.index[i]), p, marker='v', color='red',
                       s=70, zorder=6, edgecolors='darkred', linewidths=0.5)
        ax.scatter([],[],marker='v',color='red',s=70,edgecolors='darkred',
                   label=f"Swing highs ({det['pivot_highs']})")

        # Swing low markers
        for i,p in zip(sl, det['swing_low_prices']):
            ax.scatter(mdates.date2num(df.index[i]), p, marker='^', color='blue',
                       s=70, zorder=6, edgecolors='darkblue', linewidths=0.5)
        ax.scatter([],[],marker='^',color='blue',s=70,edgecolors='darkblue',
                   label=f"Swing lows ({det['pivot_lows']})")
    else:
        ws = df.iloc[det['start_idx']:det['end_idx']]
        add_trendline(ax, ws, [sl_u,ic_u], det['window'], color='red', label='Upper')
        add_trendline(ax, ws, [sl_l,ic_l], det['window'], color='blue', label='Lower')

    ep = df.loc[det['event_date'], 'Close']
    add_event_marker(ax, det['event_date'], ep, marker='D', color='orange', size=80, label='Breakout')
    ax.legend(fontsize=7, loc='upper left')
    fig.tight_layout()
    plt.show()

## 1. Triangle Detections

One chart per detection. Red ▼ = swing highs, blue ▲ = swing lows.
Trendlines run from the first pivot to just past the breakout bar.

In [ ]:
plot_detection(tri_details[0])

In [ ]:
plot_detection(tri_details[1])

In [ ]:
plot_detection(tri_details[2])

In [ ]:
plot_detection(tri_details[3])

In [ ]:
plot_detection(tri_details[4])

In [ ]:
plot_detection(tri_details[5])

In [ ]:
plot_detection(tri_details[6])

In [ ]:
plot_detection(tri_details[7])

In [ ]:
plot_detection(tri_details[8])

In [ ]:
plot_detection(tri_details[9])

In [ ]:
plot_detection(tri_details[10])

In [ ]:
plot_detection(tri_details[11])

In [ ]:
plot_detection(tri_details[12])

## 2. Channel Detections

In [ ]:
if ch_details:
    for det in ch_details:
        plot_detection(det)
else:
    print('No channels detected.')

## 3. Summary

In [ ]:
print(f'Total: {len(tri_details)} triangles, {len(ch_details)} channels')
for d in tri_details:
    print(f"  {d['event_date'].strftime('%Y-%m-%d')}  {d['pattern_type']:25s}  "
          f"cr={d.get('containment_ratio',0):.0%}  |r|=({d.get('r_upper',0):.2f},{d.get('r_lower',0):.2f})")

## Conclusion

Every triangle above shows the classic converging structure:
- Trendlines connect pivot-to-pivot with adjusted intercepts bounding the price
- 20-bar window produces compact, recognisable shapes (~1 month)
- |r| ≥ 0.9 ensures tight pivot alignment
- Types: symmetric, ascending, descending